In [3]:
import glob
import pickle

from tqdm import tqdm

"""Data module for the tactic generator."""
import pickle
from pathlib import Path
import random
from typing import Optional

import lightning.pytorch as pl
import torch
from loguru import logger
from pymongo import MongoClient
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

from experiments.end_to_end.lightning_common import Batch
from experiments.end_to_end.process_traces import add_rand_idx, filter_traces
from experiments.end_to_end.proof_node import ErrorNode, Status
from experiments.end_to_end.stream_dataset import GoalStreamDataset, worker_init_fn



In [4]:
database='lean_e2e'
collection='intern_lm_transitions_top32'
replace='keep'
host='localhost:27017'  # mongodb host

# batch_size = batch_size
# eval_batch_size = eval_batch_size
# max_seq_len = max_seq_len
# num_workers = num_workers
# tokenizer = AutoTokenizer.from_pretrained(model_name)

fields = ['goal', 'tactic', 'result', 'theorem', 'status', 'time']



paths = {
    # 'dpp_4': "../runs/internlm/dpp_critic/2025_03_08/02_33_49/traces/0/",
    # 'dpp_3': "../runs/internlm/dpp_critic/2025_03_05/18_40_59/traces/0/",
    # 'dpp_2_1': "../runs/internlm/dpp_critic/2025_03_03/12_57_34/traces/0/",
    # 'cg_1': "../runs/internlm/internlm_cg/2025_03_03/11_50_33/traces/0/",
    # 'cg_2': "../runs/internlm/internlm_cg/2025_03_08/05_15_27/traces/0/",
    # 'cg_3': "../runs/internlm/internlm_cg/2025_03_06/04_23_46/traces/0/",
    # 'bfs_1': "../runs/internlm/internlm/2025_03_15/15_49_16/traces/0/",
    'bfs_2': "../runs/internlm_top32/traces/0/",
    # 'dpp_2_2': "../runs/internlm/dpp_critic_k_2/2025_03_19/10_20_24/traces/0/",
    # 'dpp_2_3': "../runs/internlm/dpp_critic_k_2/2025_03_24/10_42_57/traces/0/"
}

db = MongoClient()[database]
# print (type(database), type(collection))

path = Path(paths['bfs_2'])
trace_files = [x for x in path.rglob("*") if x.is_file()]


collection = MongoClient()[database][collection]

# split val set at the node level rather than trace level
def add_trace(trace, split_size):
    nodes = trace.nodes
    nodes[trace.tree.goal] = trace.tree

    random.shuffle(trace.trace)

    for edge in trace.trace[:int(split_size * len(trace.trace))]:
        split = 'train'
        data = {'goal': edge.src.goal, 'tactic': edge.tactic,
                'logprob': edge.tac_logprob,
                'split': split,
                'theorem': trace.theorem.full_name,
                'time': edge.time}
        if len(edge.dst) == 1 and isinstance(edge.dst[0], ErrorNode):
            data['result'] = edge.dst[0].inner.message.split(' tactic_state')[0]
            data['status'] = 'failed'
            # skip with 0.9 probability if simple error
            if 'expected end of input' in data['result']:
                if random.random() < 0.9:
                    continue
        else:
            data['result'] = ''.join([d.goal if hasattr(d, 'goal') else 'Proven' for d in edge.dst])
            data['status'] = 'success'

        collection.insert_one(data)

    for edge in trace.trace[int(split_size * len(trace.trace)):]:
        split = 'val'
        data = {'goal': edge.src.goal, 'tactic': edge.tactic,
                'logprob': edge.tac_logprob,
                'split': split,
                'theorem': trace.theorem.full_name,
                'time': edge.time}
        if len(edge.dst) == 1 and isinstance(edge.dst[0], ErrorNode):
            data['result'] = edge.dst[0].inner.message.split(' tactic_state')[0]
            data['status'] = 'failed'
        else:
            data['result'] = ''.join([d.goal if hasattr(d, 'goal') else 'Proven' for d in edge.dst])
            data['status'] = 'success'

        collection.insert_one(data)

# logger.info('Processing traces for training transition model...')
for trace in tqdm(trace_files):
    try:
        trace = pickle.load(open(trace, 'rb'))
    except Exception as e:
        logger.info(f'Error loading {trace}: {e}')
        continue
    if isinstance(trace.tree, ErrorNode):
        continue

    add_trace(trace, 0.95)

# splitting val set based on files
# def add_trace(trace, split):
#     nodes = trace.nodes
#     nodes[trace.tree.goal] = trace.tree
#
#     for edge in trace.trace:
#         data = {'goal': edge.src.data['augmented_state'], 'tactic': edge.tactic,
#                 'logprob': edge.tac_logprob,
#                 'split': split,
#                 'theorem': trace.theorem.full_name,
#                 'time': edge.time}
#         if len(edge.dst) == 1 and isinstance(edge.dst[0], ErrorNode):
#             data['result'] = edge.dst[0].inner.message.split(' tactic_state')[0]
#             data['status'] = 'failed'
#         else:
#             data['result'] = ''.join([d.goal if hasattr(d, 'goal') else 'Proven' for d in edge.dst])
#             data['status'] = 'success'
#
#         collection.insert_one(data)
#
# logger.info('Processing traces for training transition model...')
# for trace in tqdm(trace_files[:int(0.9 * len(trace_files))]):
#     trace = pickle.load(open(trace, 'rb'))
#     if isinstance(trace.tree, ErrorNode):
#         continue
#
#     add_trace(trace, 'train')
#
# logger.info('Processing traces for validating transition model...')
# for trace in tqdm(trace_files[int(0.9 * len(trace_files)):]):
#     trace = pickle.load(open(trace, 'rb'))
#     if isinstance(trace.tree, ErrorNode):
#         continue
#
#     add_trace(trace, 'val')
#
add_rand_idx(collection)
## val


100%|██████████| 245/245 [00:21<00:00, 11.62it/s]
